In [21]:
# ==========================================
# 삼성전자 주가 예측 (LSTM)
# ==========================================

!pip install yfinance

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, SimpleRNN
from tensorflow.keras.callbacks import EarlyStopping


In [22]:
df = yf.download(
    "005930.KS",
    start="2018-01-01",
    end="2026-07-01",
    auto_adjust=True
)

df.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,005930.KS,005930.KS,005930.KS,005930.KS,005930.KS
Date,,,,,
2018-01-02,41206.007812,41512.912614,41012.173201,41496.759730,8474250
2018-01-03,41690.593750,42449.779301,41529.064909,42433.626417,10013500
2018-01-04,41254.460938,42142.869454,40899.097531,42094.410808,11695450
2018-01-05,42094.398438,42094.398438,41351.366078,41432.130465,9481150
2018-01-08,42013.648438,42417.470510,41593.673482,42320.553213,8383650


In [23]:
close = df[["Close"]]

close.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2080 entries, 2018-01-02 to 2026-06-30
Data columns (total 1 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   (Close, 005930.KS)  2080 non-null   float64
dtypes: float64(1)
memory usage: 32.5 KB


In [24]:
# 정규화 

# 최소값 = 100
# 최대값 = 190

# 현재값−100
#------------
#   190−100
scaler = MinMaxScaler() # StandardScaler
scaled = scaler.fit_transform(close)

scaled

array([[0.03016294],
       [0.03162568],
       [0.0303092 ],
       ...,
       [0.9305737 ],
       [0.88076788],
       [0.91397176]])

In [25]:
# 시계열 데이터 생성
# 60일 -> 다음날

# ==========================================
# data_size = -1
# time_steps = 60
# features = 1
# ==========================================

window_size = 60

x = []
y = []

for i in range(window_size,len(scaled)):
    x.append(scaled[i - window_size : i])
    y.append(scaled[i])
    #print(window_size,i)

x = np.array(x)
y = np.array(y)

#(2020, 60, 1) (2020, 1)
print(x.shape, y.shape)

(2020, 60, 1) (2020, 1)


In [26]:
# train/ test 테이터 나누기

train_size = int(len(x) * 0.8)
train_size

x_train = x[:train_size]
x_test = x[train_size:]

y_train = y[:train_size]
y_test = y[train_size:]

print(x_train.shape)
print(x_test.shape)


(1616, 60, 1)
(404, 60, 1)


In [ ]:
# 모델링

model = Sequential([
    Input(shape=(window_size,1)),
    SimpleRNN(64,return_sequences=True),
    SimpleRNN(32),
    Dense(16,activation="relu"),
    Dense(1)
])

model.summary()

# model = Sequential([
#     Input(shape=(window_size,1)),
#     LSTM(64,return_sequences=True),
#     LSTM(32),
#     Dense(16,activation="relu"),
#     Dense(1)
# ])

# 컴파일
model.compile(
    optimizer="adam",
    loss ="mse", 
    metrics=["mae"] # 평균 절대 오차 # 선형회귀 이기 때문에 accurcy 아님
)

# EarlyStopping
es = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

# 학습
history = model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[es]
)

# 예측
pred = model.predict(x_test)

# 원래 가격 복원

pred_price = scaler.inverse_transform(pred)
real_price = scaler.inverse_transform(y_test)



Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_4 (SimpleRNN)        │ (None, 60, 64)         │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_5 (SimpleRNN)        │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,873 (30.75 KB)

 Trainable params: 7,873 (30.75 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100


In [ ]:
# ------------------------------------------
# 그래프
# ------------------------------------------

plt.figure(figsize=(15,6))

plt.plot(real_price, label="Real")
plt.plot(pred_price, label="Predict")

plt.title("Samsung Electronics Stock Prediction")
plt.xlabel("Time")
plt.ylabel("Price")

plt.legend()

plt.show()